In [ ]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0 = all logs, 1 = INFO, 2 = WARNING, 3 = ERROR only
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import tensorflow as tf

In [ ]:
import sys
# Force early GPU initialization
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        with tf.device('/GPU:0'):
            _ = tf.constant([1.0])
    except RuntimeError as e:
        print(f"RuntimeError during GPU warm-up: {e}")

print("\n" + "="*60)
print("TENSORFLOW GPU TEST RESULTS")
print("="*60)

# TensorFlow version
print(f"TensorFlow Version: {tf.__version__}")

# Python version
print(f"Python Version: {sys.version.split()[0]}")

# CUDA and cuDNN support (inferred)
build_info = tf.sysconfig.get_build_info()
cuda_version = build_info.get('cuda_version', 'Unknown')
cudnn_version = build_info.get('cudnn_version', 'Unknown')

print(f"CUDA Version (from build): {cuda_version}")
print(f"cuDNN Version (from build): {cudnn_version}")

# GPU status
gpu_available = len(gpus) > 0
print(f"\nGPU Available: {gpu_available}")

if gpu_available:
    for idx, gpu in enumerate(gpus):
        print(f"GPU {idx}: {gpu.name}")
else:
    print("No GPU devices detected. Defaulting to CPU.")

# Matrix multiplication test
print(f"\nRunning matrix multiplication test...")
with tf.device('/GPU:0' if gpu_available else '/CPU:0'):
    a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
    b = tf.constant([[1.0, 0.0], [0.0, 1.0]])
    c = tf.matmul(a, b)

print(f"Result:")
print(c.numpy())
print("\nTest completed successfully.")
print("="*60)

In [ ]:
import requests
import pandas as pd
import tensorflow as tf
import os
import random
import numpy as np

In [ ]:
# Define the URL and target path
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
save_path = "/home/jwens/PycharmProjects/School-Projects/Graduate Projects - MS AI and Machine Learning/CSC580 Capstone/data/Iris/iris.data"

# Ensure the directory exists
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# Download and save the file
r = requests.get(url)
with open(save_path, 'wb') as f:
    f.write(r.content)

In [ ]:
# Load Iris dataset with defined column names
iris_columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "class"]
iris_df = pd.read_csv(save_path, header=None, names=iris_columns)


In [ ]:
# Encode class labels as integers
label_map = {
    "Iris-setosa": 0,
    "Iris-versicolor": 1,
    "Iris-virginica": 2
}
iris_df["class"] = iris_df["class"].map(label_map)
iris_df.head(3)

In [ ]:
# Shuffle the dataset
iris_df = iris_df.sample(frac=1.0, random_state=4321)

# Separate features and labels
x = iris_df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = iris_df["class"]

# Center features by subtracting the mean
x = x - x.mean(axis=0)

# One-hot encode labels
y = tf.one_hot(y, depth=3)
y = tf.cast(y, tf.float32)
print(x)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras import backend as K

K.clear_session()
# Define the neural network model
model = Sequential([
    Dense(32, activation='relu', input_shape=(4,)),  # Input layer (4 features) + 32-node hidden layer
    Dense(16, activation='relu'),                    # Second hidden layer
    Dense(3, activation='softmax')                   # Output layer for 3 classes
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.summary()

In [ ]:
history = model.fit(
    x,                # Centered feature matrix
    y,                # One-hot encoded labels
    batch_size=64,
    epochs=25,
    verbose=1         # Shows progress bar per epoch
)

In [ ]:
def fix_random_seed(seed):
    try:
        np.random.seed(seed)
    except NameError:
        print("Warning: Numpy is not imported. Setting the seed for Numpy failed.")
    try:
        tf.random.set_seed(seed)
    except NameError:
        print("Warning: TensorFlow is not imported. Setting the seed for TensorFlow failed.")
    try:
        random.seed(seed)
    except NameError:
        print("Warning: random module is not imported. Setting the seed for random failed.")

# Fixing the random seed
fix_random_seed(4321)


In [ ]:
#Functional API
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model

In [ ]:
inp1 = Input(shape=(4,))  # raw iris features (e.g., petal and sepal)
inp2 = Input(shape=(2,))  # PCA features (2 principal components)
out1 = Dense(16, activation='relu')(inp1)
out2 = Dense(16, activation='relu')(inp2)

In [ ]:
concat = Concatenate(axis=1)
out = concat([out1, out2])
out = Dense(16, activation='relu')(out)
out = Dense(3, activation='softmax')(out)
model = Model(inputs=[inp1, inp2], outputs=out)
model.compile(loss='categorical_crossentropy', optimizer='adam',
metrics=['acc'])
model.summary()

In [ ]:
tf.keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

In [ ]:
from sklearn.decomposition import PCA

# Compute the first two principal components of x
pca_model = PCA(n_components=2, random_state=4321)
x_pca = pca_model.fit_transform(x)
print(x_pca)


In [ ]:
model.fit([x, x_pca], y, batch_size=64, epochs=10)

In [ ]:
def __init__(self, units=32, activation=None):
    super(MuLBiasDense, self).__init__()
    self.units = units
    self.activation = activation


In [ ]:
def build(self, input_shape):
    self.w = self.add_weight(shape=(input_shape[-1], self.units),
                             initializer='glorot_uniform',
                             trainable=True)
    self.b = self.add_weight(shape=(self.units,),
                             initializer='glorot_uniform',
                             trainable=True)
    self.b_mul = self.add_weight(shape=(self.units,),
                                 initializer='glorot_uniform',
                                 trainable=True)


In [ ]:
def call(self, inputs):
    out = (tf.matmul(inputs, self.w) + self.b) * self.b_mul
    return layers.Activation(self.activation)(out)


In [ ]:
import os
import tensorflow as tf

data_dir = "/home/jwens/PycharmProjects/School-Projects/Graduate Projects - MS AI and Machine Learning/CSC580 Capstone/data/Flower Color Images"

csv_ds = tf.data.experimental.CsvDataset(
    os.path.join(data_dir, 'flower_labels.csv'),
    record_defaults=("", -1),
    header=True
)

for item in csv_ds.take(5):
    print(item)


In [ ]:
fname_ds = csv_ds.map(lambda a,b: a)
label_ds = csv_ds.map(lambda a,b: b)

In [ ]:
def get_image(file_path):
    img = tf.io.read_file(tf.strings.join([data_dir, "/", file_path]))
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)
    return tf.image.resize(img, [64, 64])


In [ ]:
image_ds = fname_ds.map(get_image)

In [ ]:
label_ds = label_ds.map(lambda x: tf.one_hot(x, depth=10))

In [ ]:
data_ds = tf.data.Dataset.zip((image_ds, label_ds))

In [ ]:
for item in data_ds:
    print(item)


In [ ]:
data_ds = data_ds.shuffle(buffer_size= 20)

In [ ]:
data_ds = data_ds.batch(5)

In [ ]:
for item in data_ds:
    print(item)
    break


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense

model = Sequential([
    Conv2D(64, (5,5), activation='relu', input_shape=(64,64,3)),
    Flatten(),
    Dense(10, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['acc'])
odel.fit(data_ds, epochs=10)

In [ ]:
data_dir = "/home/jwens/PycharmProjects/School-Projects/Graduate Projects - MS AI and Machine Learning/CSC580 Capstone/data/Flower Color Images"


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
img_gen = ImageDataGenerator()

In [ ]:
labels_df = pd.read_csv(os.path.join(data_dir, 'flower_labels.csv'), header=0)
gen_iter = img_gen.flow_from_dataframe(
    dataframe=labels_df,
    directory=data_dir,
    x_col='file',
    y_col='label',
    class_mode='raw',
    batch_size=5,
    target_size=(64,64)
)

In [ ]:
for item in gen_iter:
    print(item)
    break

In [ ]:
import tensorflow_datasets as tfds

In [ ]:
tfds.list_builders()

In [ ]:
data, info = tfds.load("cifar10", with_info=True)

In [ ]:
print(info)

In [ ]:
print(data)

In [ ]:
train_ds = data["train"]

In [ ]:
train_ds = data["train"].batch(16)

In [ ]:
for item in train_ds:
    print(item)
    break


In [ ]:
def format_data(image, label):
    label = tf.cast(label, tf.int32)
    label = tf.one_hot(label, depth=10)
    label = tf.cast(label, tf.float32)
    return image, label


train_ds = train_ds.map(format_data)

def resize_images(image, label):
    image = tf.image.resize(image, [64, 64])
    return image, label

train_ds = train_ds.map(resize_images)


In [ ]:
for img, lbl in train_ds.take(1):
    print("Image shape:", img.shape)
    print("Label shape:", lbl.shape, "dtype:", lbl.dtype)
